In [1]:
!pip install openai-whisper googletrans==4.0.0-rc1 gTTS librosa soundfile gradio

In [ ]:
import whisper
import librosa
import numpy as np
from googletrans import Translator
from gtts import gTTS
import gradio as gr
import os

# Yahan hum Whisper Model (Kaan) ko load kar rahe hain
print("Whisper model taiyar ho rha hai... Thoda samay lag sakta hai.")
model = whisper.load_model("base")
translator = Translator()

# Yahan Dusra Worker (Librosa) mood detect karne ka logic samajh raha hai
def detect_emotion(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=None)
        pitches = librosa.yin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
        pitches = pitches[~np.isnan(pitches)]
        pitches = pitches[pitches > 0]

        if len(pitches) == 0:
            return "Neutral / Calm 😐"

        mean_pitch = np.mean(pitches)

        # Aawaz ki pitch ke hisaab se emotion tay hota hai
        if mean_pitch > 220:
            return "Happy / Excited 😊🎉"
        elif mean_pitch > 160:
            return "Angry / High Energy 😠🔥"
        elif mean_pitch < 120:
            return "Sad / Low Energy 😢🌧️"
        else:
            return "Neutral / Calm 😐"
    except Exception as e:
        return "Neutral / Calm 😐"

# Main Engine - Jo saare kaam ek ke baad ek karega
def process_audio(audio_path, target_lang):
    if audio_path is None:
        return "Kripya audio upload ya record karein.", None, None, None

    # 1. Speech-to-Text (Aawaz se text banana)
    result = model.transcribe(audio_path)
    original_text = result['text']

    # 2. Emotion Detection (Mood pehchanna)
    detected_emotion = detect_emotion(audio_path)

    # Bhasha ka code chunna (Language Preference)
    lang_map = {
        "Hindi": "hi", "English": "en", "Spanish": "es",
        "French": "fr", "German": "de", "Tamil": "ta",
        "Telugu": "te", "Marathi": "mr"
    }
    lang_code = lang_map.get(target_lang, "hi")

    # 3. Translation (Bhasha badalna)
    translated = translator.translate(original_text, dest=lang_code)
    translated_text = translated.text

    # 4. Text-to-Speech (Translated text ko naya audio banana)
    tts = gTTS(text=translated_text, lang=lang_code, slow=False)
    output_audio_path = "translated_output.mp3"
    tts.save(output_audio_path)

    return original_text, detected_emotion, translated_text, output_audio_path

# Gradio UI Layout - Sundar sa web page banane ke liye
languages = ["Hindi", "English", "Spanish", "French", "German", "Tamil", "Telugu", "Marathi"]

interface = gr.Interface(
    fn=process_audio,
    inputs=[
        gr.Audio(sources=["microphone", "upload"], type="filepath", label="Apna Audio Record karein ya File Upload karein"),
        gr.Dropdown(choices=languages, value="Hindi", label="Kis bhasha (Language) mein translate karna hai?")
    ],
    outputs=[
        gr.Textbox(label="Aapne kya bola (Original Text)"),
        gr.Textbox(label="Detected Emotion (Aawaz ka Bhaav)"),
        gr.Textbox(label="Translated Text (Anuvaad)"),
        gr.Audio(label="Translated Audio (Nayi Aawaz)", type="filepath")
    ],
    title="🎙️ EmoVoice: Audio-to-Audio Translator cum Emotion Detector",
    description="Yeh AI tool aapke audio ka emotion detect karega aur sath hi aapki pasandida bhasha mein use translate karke sunayega!"
)

# App ko Live start karna
interface.launch(share=True, debug=True)

Whisper model taiyar ho rha hai... Thoda samay lag sakta hai.


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 106MiB/s]


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
IMPORTANT: You are using gradio version 4.19.1, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://816da625b7926410e1.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
